# NumPy for Machine Learning — Notebook 3 of 3

## Table of Contents
1. Data Preprocessing
2. Feature Engineering
3. Matrix Operations for ML
4. Weight Initialization
5. Activation Functions
6. Loss Functions
7. Regularization
8. Linear Regression from Scratch
9. Logistic Regression from Scratch
10. Backpropagation — Step by Step
11. Dropout
12. Batch Normalization
13. Adam Optimizer
14. SVD for Dimensionality Reduction
15. PCA from Scratch
16. Evaluation Metrics from Scratch
17. Image as NumPy Array
18. einsum for ML
19. Gradient Checking
20. Data Splitting
21. Vectorization Benchmark
22. Cheat Sheet
23. Exercises

**Note:** Implements real ML from scratch using only NumPy

## Section 1: Data Preprocessing
Why preprocessing? Raw data has different scales — must normalize

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Create a synthetic dataset with 4 features at different scales
X = np.random.randn(100, 4) * np.array([100, 10, 1, 0.1]) + np.array([50, -10, 0, 5])
print('Original Data (first 5 rows):\n', X[:5])

Min-Max Normalization — formula: (x - min) / (max - min) → scale to [0,1]

In [ ]:
np.random.seed(42)
X_min = np.min(X, axis=0)
X_max = np.max(X, axis=0)
X_norm = (X - X_min) / (X_max - X_min)
print('Normalized Data (first 5 rows):\n', X_norm[:5])

Z-score Standardization — formula: (x - mean) / std → mean=0, std=1

In [ ]:
np.random.seed(42)
X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0)
X_std_scaled = (X - X_mean) / X_std
print('Standardized Data (first 5 rows):\n', X_std_scaled[:5])

Compare original vs normalized vs standardized with 3-panel matplotlib plot

In [ ]:
np.random.seed(42)
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].scatter(X[:, 0], X[:, 1], alpha=0.5)
axs[0].set_title('Original Data')
axs[1].scatter(X_norm[:, 0], X_norm[:, 1], alpha=0.5, color='orange')
axs[1].set_title('Min-Max Normalized')
axs[2].scatter(X_std_scaled[:, 0], X_std_scaled[:, 1], alpha=0.5, color='green')
axs[2].set_title('Z-score Standardized')
plt.show()

np.clip() for outlier removal — clip extreme values to [p5, p95] percentile

In [ ]:
np.random.seed(42)
X_with_outliers = np.append(X, [[1000, 1000, 1000, 1000]], axis=0)
p5, p95 = np.percentile(X_with_outliers, [5, 95], axis=0)
X_clipped = np.clip(X_with_outliers, p5, p95)
print('Max before clip:', np.max(X_with_outliers, axis=0))
print('Max after clip:', np.max(X_clipped, axis=0))

Handling NaN — np.isnan() + replace with column mean

In [ ]:
np.random.seed(42)
X_nan = X.copy()
X_nan[np.random.randint(0, 100, 5), np.random.randint(0, 4, 5)] = np.nan

# Replace with column mean
col_means = np.nanmean(X_nan, axis=0)
inds = np.where(np.isnan(X_nan))
X_nan[inds] = np.take(col_means, inds[1])
print('Any NaNs left?', np.isnan(X_nan).any())

Handling NaN — remove rows with any NaN using boolean indexing

In [ ]:
np.random.seed(42)
X_nan2 = X.copy()
X_nan2[np.random.randint(0, 100, 5), np.random.randint(0, 4, 5)] = np.nan

mask = ~np.isnan(X_nan2).any(axis=1)
X_clean = X_nan2[mask]
print(f'Rows before: {X_nan2.shape[0]}, Rows after: {X_clean.shape[0]}')

## Section 2: Feature Engineering
Code cells for creating new features

Add bias column — np.hstack([np.ones((n,1)), X])

In [ ]:
np.random.seed(42)
bias_col = np.ones((X.shape[0], 1))
X_with_bias = np.hstack([bias_col, X])
print('Shape with bias:', X_with_bias.shape)

Polynomial features — for degree 2: [x1, x2, x1^2, x2^2, x1*x2]

In [ ]:
np.random.seed(42)
x1 = X[:, 0:1]
x2 = X[:, 1:2]
X_poly = np.hstack([x1, x2, x1**2, x2**2, x1*x2])
print('Shape with polynomial features:', X_poly.shape)

One-hot encoding from scratch using np.eye(num_classes)[labels]

In [ ]:
np.random.seed(42)
labels = np.random.randint(0, 3, 10)
num_classes = 3
one_hot = np.eye(num_classes)[labels]
print('Labels:', labels)
print('One-hot encoded:\n', one_hot)

Feature binning — np.digitize() to bin continuous feature into categories

In [ ]:
np.random.seed(42)
bins = np.array([0, 25, 50, 75, 100])
feature = np.random.randint(-10, 110, 10)
binned = np.digitize(feature, bins)
print('Feature:', feature)
print('Binned:', binned)

Log transform — np.log1p() for right-skewed features

In [ ]:
np.random.seed(42)
skewed_feature = np.random.exponential(scale=2.0, size=10)
log_transformed = np.log1p(skewed_feature)
print('Original skewed:', skewed_feature)
print('Log transformed:', log_transformed)

Feature interaction — multiply two feature columns element-wise

In [ ]:
np.random.seed(42)
feature1 = X[:, 0]
feature2 = X[:, 1]
interaction = feature1 * feature2
print('Interaction feature shape:', interaction.shape)

## Section 3: Matrix Operations for ML
Fundamental matrix operations used throughout machine learning.

Design matrix X — shape (n_samples, n_features) with bias

In [ ]:
np.random.seed(42)
n_samples, n_features = 10, 3
X_design = np.hstack([np.ones((n_samples, 1)), np.random.randn(n_samples, n_features)])
print('Design matrix shape:', X_design.shape)

Gram matrix — X.T @ X — used in kernel methods, normal equation

In [ ]:
np.random.seed(42)
gram_matrix = X_design.T @ X_design
print('Gram matrix shape:', gram_matrix.shape)

Euclidean distance matrix from scratch — between all pairs of points
Use broadcasting: dist[i,j] = ||x_i - x_j||_2
Formula: ||a-b||^2 = ||a||^2 + ||b||^2 - 2*a.b

In [ ]:
np.random.seed(42)
A = np.random.randn(5, 2)
B = np.random.randn(4, 2)
# ||A-B||^2 = ||A||^2 + ||B||^2 - 2 A.B^T
A_sq = np.sum(A**2, axis=1, keepdims=True)
B_sq = np.sum(B**2, axis=1)
dist_sq = A_sq + B_sq - 2 * (A @ B.T)
dist = np.sqrt(np.maximum(dist_sq, 0)) # prevent negative due to floating point
print('Distance matrix shape:', dist.shape)

Cosine similarity matrix from scratch

In [ ]:
np.random.seed(42)
norm_A = np.linalg.norm(A, axis=1, keepdims=True)
norm_B = np.linalg.norm(B, axis=1, keepdims=True)
cos_sim = (A @ B.T) / (norm_A @ norm_B.T)
print('Cosine similarity matrix:\n', cos_sim)

Covariance matrix — np.cov(X.T)

In [ ]:
np.random.seed(42)
cov_matrix = np.cov(X_design[:, 1:].T)
print('Covariance matrix shape:', cov_matrix.shape)

## Section 4: Weight Initialization
Why initialization matters — wrong init → vanishing/exploding gradients
Show the problem: if weights too large → explode, too small → vanish

Random initialization (naive) — np.random.randn(fan_in, fan_out) * 0.01

In [ ]:
np.random.seed(42)
fan_in, fan_out = 100, 50
W_naive = np.random.randn(fan_in, fan_out) * 0.01
print('Naive init variance:', np.var(W_naive))

Xavier/Glorot initialization — std = sqrt(2 / (fan_in + fan_out))
Used for Sigmoid/Tanh activations

In [ ]:
np.random.seed(42)
W_xavier = np.random.randn(fan_in, fan_out) * np.sqrt(2 / (fan_in + fan_out))
print('Xavier init variance:', np.var(W_xavier))

He initialization — std = sqrt(2 / fan_in)
Used for ReLU activations

In [ ]:
np.random.seed(42)
W_he = np.random.randn(fan_in, fan_out) * np.sqrt(2 / fan_in)
print('He init variance:', np.var(W_he))

Zeros initialization — why it fails (symmetry breaking problem)

In [ ]:
np.random.seed(42)
W_zeros = np.zeros((fan_in, fan_out))
print('Zeros init variance:', np.var(W_zeros))

Matplotlib — plot distribution of activations with different initializations
Show how Xavier and He keep activations in good range

In [ ]:
np.random.seed(42)
inputs = np.random.randn(1000, fan_in)
act_naive = inputs @ W_naive
act_xavier = inputs @ W_xavier
act_he = inputs @ W_he

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].hist(act_naive.flatten(), bins=30); axs[0].set_title('Naive')
axs[1].hist(act_xavier.flatten(), bins=30); axs[1].set_title('Xavier')
axs[2].hist(act_he.flatten(), bins=30); axs[2].set_title('He')
plt.show()

## Section 5: Activation Functions
Activations add non-linearity — without them, neural network = linear model

Sigmoid — σ(x) = 1 / (1 + exp(-x)), derivative = σ(x)(1 - σ(x))
Implement forward AND derivative functions

In [ ]:
np.random.seed(42)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

ReLU — max(0, x), derivative = (x > 0).astype(float)
Implement forward AND derivative

In [ ]:
np.random.seed(42)
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

Leaky ReLU — max(α*x, x), derivative
Implement forward AND derivative

In [ ]:
np.random.seed(42)
def leaky_relu(x, alpha=0.01):
    return np.maximum(alpha * x, x)

def leaky_relu_derivative(x, alpha=0.01):
    dx = np.ones_like(x)
    dx[x <= 0] = alpha
    return dx

Tanh — (exp(x) - exp(-x)) / (exp(x) + exp(-x)), derivative = 1 - tanh^2(x)
Implement forward AND derivative

In [ ]:
np.random.seed(42)
def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - np.tanh(x)**2

Softmax — exp(x_i) / sum(exp(x)) with NUMERICAL STABILITY trick: subtract max first
Implement forward function

In [ ]:
np.random.seed(42)
def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=-1, keepdims=True)

Plot ALL activations and their derivatives in a 2x3 grid — matplotlib

In [ ]:
np.random.seed(42)
x_vals = np.linspace(-5, 5, 100)
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
axs[0,0].plot(x_vals, sigmoid(x_vals)); axs[0,0].set_title('Sigmoid')
axs[0,1].plot(x_vals, relu(x_vals)); axs[0,1].set_title('ReLU')
axs[0,2].plot(x_vals, leaky_relu(x_vals)); axs[0,2].set_title('Leaky ReLU')
axs[1,0].plot(x_vals, tanh(x_vals)); axs[1,0].set_title('Tanh')
axs[1,1].plot(x_vals, sigmoid_derivative(x_vals)); axs[1,1].set_title('Sigmoid Deriv')
axs[1,2].plot(x_vals, relu_derivative(x_vals)); axs[1,2].set_title('ReLU Deriv')
plt.tight_layout()
plt.show()

## Section 6: Loss Functions
Loss measures how wrong predictions are

MSE — mean((y_pred - y_true)^2), gradient = 2*(y_pred - y_true)/n
Implement loss AND gradient functions

In [ ]:
np.random.seed(42)
def mse_loss(y_pred, y_true):
    return np.mean((y_pred - y_true)**2)

def mse_gradient(y_pred, y_true):
    return 2 * (y_pred - y_true) / y_pred.size

MAE — mean(|y_pred - y_true|), gradient = sign(y_pred - y_true)/n
Implement loss AND gradient

In [ ]:
np.random.seed(42)
def mae_loss(y_pred, y_true):
    return np.mean(np.abs(y_pred - y_true))

def mae_gradient(y_pred, y_true):
    return np.sign(y_pred - y_true) / y_pred.size

RMSE — sqrt(MSE)

In [ ]:
np.random.seed(42)
def rmse_loss(y_pred, y_true):
    return np.sqrt(mse_loss(y_pred, y_true))

Binary Cross-Entropy — -mean(y*log(p) + (1-y)*log(1-p))
Use np.clip(p, 1e-7, 1-1e-7) to avoid log(0)!
Implement loss AND gradient

In [ ]:
np.random.seed(42)
def bce_loss(y_pred, y_true):
    p = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

def bce_gradient(y_pred, y_true):
    p = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return (p - y_true) / (p * (1 - p) * y_pred.size)

Categorical Cross-Entropy — -mean(sum(y_true * log(y_pred)))
Implement with softmax output

In [ ]:
np.random.seed(42)
def cce_loss(y_pred, y_true):
    p = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(np.sum(y_true * np.log(p), axis=-1))

Hinge loss — max(0, 1 - y*y_pred) for SVM

In [ ]:
np.random.seed(42)
def hinge_loss(y_pred, y_true):
    return np.mean(np.maximum(0, 1 - y_true * y_pred))

Matplotlib — plot all loss functions as curves

In [ ]:
np.random.seed(42)
y_true_val = 1
y_preds = np.linspace(0.01, 0.99, 100)
bce_vals = [-np.log(p) for p in y_preds] # For true class = 1
mse_vals = [(p - 1)**2 for p in y_preds]
mae_vals = [abs(p - 1) for p in y_preds]
hinge_vals = [np.maximum(0, 1 - 1 * p) for p in np.linspace(-1, 2, 100)]

plt.figure(figsize=(10, 6))
plt.plot(y_preds, bce_vals, label='BCE (y_true=1)')
plt.plot(y_preds, mse_vals, label='MSE')
plt.plot(y_preds, mae_vals, label='MAE')
plt.plot(np.linspace(-1, 2, 100), hinge_vals, label='Hinge (y_true=1)')
plt.legend()
plt.title('Loss Functions')
plt.show()

## Section 7: Regularization
Regularization prevents overfitting by penalizing large weights

L2 (Ridge) regularization term — lambda * np.sum(W**2)
L2 gradient — lambda * 2 * W
Effect: pushes weights toward 0 (weight decay)

In [ ]:
np.random.seed(42)
def l2_reg(W, lam=0.01):
    return lam * np.sum(W**2)

def l2_gradient(W, lam=0.01):
    return lam * 2 * W

L1 (Lasso) regularization term — lambda * np.sum(np.abs(W))
L1 gradient — lambda * np.sign(W)
Effect: pushes weights exactly to 0 (feature selection)

In [ ]:
np.random.seed(42)
def l1_reg(W, lam=0.01):
    return lam * np.sum(np.abs(W))

def l1_gradient(W, lam=0.01):
    return lam * np.sign(W)

Elastic Net — combination of L1 + L2

In [ ]:
np.random.seed(42)
def elastic_net_reg(W, lam1=0.01, lam2=0.01):
    return l1_reg(W, lam1) + l2_reg(W, lam2)

Visualize L1 vs L2 penalty surface using meshgrid and contour plot

In [ ]:
np.random.seed(42)
w1 = np.linspace(-2, 2, 100)
w2 = np.linspace(-2, 2, 100)
W1, W2 = np.meshgrid(w1, w2)
L1_penalty = np.abs(W1) + np.abs(W2)
L2_penalty = W1**2 + W2**2

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].contourf(W1, W2, L1_penalty, levels=20)
axs[0].set_title('L1 Penalty Surface')
axs[1].contourf(W1, W2, L2_penalty, levels=20)
axs[1].set_title('L2 Penalty Surface')
plt.show()

## Section 8: Linear Regression from Scratch
The simplest ML model — predicts continuous values

Generate synthetic dataset — y = 2*x1 + 3*x2 + noise

In [ ]:
np.random.seed(42)
X_lin = np.random.randn(100, 2)
y_lin = 2 * X_lin[:, 0] + 3 * X_lin[:, 1] + np.random.randn(100) * 0.5

Normal Equation — theta = (X^T X)^{-1} X^T y (closed form solution)
Use np.linalg.inv or np.linalg.lstsq

In [ ]:
np.random.seed(42)
X_lin_bias = np.hstack([np.ones((100, 1)), X_lin])
theta_normal = np.linalg.inv(X_lin_bias.T @ X_lin_bias) @ X_lin_bias.T @ y_lin
print('Theta from Normal Equation:', theta_normal)

Predict and compute MSE with normal equation

In [ ]:
np.random.seed(42)
y_pred_normal = X_lin_bias @ theta_normal
mse_normal = mse_loss(y_pred_normal, y_lin)
print('MSE (Normal Equation):', mse_normal)

Gradient Descent implementation — step-by-step:
Initialize weights, compute predictions, compute loss, compute gradient, update weights

In [ ]:
np.random.seed(42)
theta_gd = np.zeros(3)
lr = 0.1
# single step
y_pred_step = X_lin_bias @ theta_gd
grad_step = (2/100) * X_lin_bias.T @ (y_pred_step - y_lin)
theta_gd -= lr * grad_step

Training loop — run gradient descent for n_epochs, record loss history

In [ ]:
np.random.seed(42)
theta_gd = np.zeros(3)
epochs = 50
loss_history = []
for i in range(epochs):
    y_pred = X_lin_bias @ theta_gd
    loss = mse_loss(y_pred, y_lin)
    loss_history.append(loss)
    grad = (2/100) * X_lin_bias.T @ (y_pred - y_lin)
    theta_gd -= lr * grad
print('Final Theta (GD):', theta_gd)

Plot loss curve — matplotlib

In [ ]:
np.random.seed(42)
plt.plot(loss_history)
plt.title('Linear Regression Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.show()

Plot: actual vs predicted values

In [ ]:
np.random.seed(42)
plt.scatter(y_lin, X_lin_bias @ theta_gd)
plt.plot([y_lin.min(), y_lin.max()], [y_lin.min(), y_lin.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted')
plt.show()

## Section 9: Logistic Regression from Scratch
Binary classification — output is probability via sigmoid

Generate synthetic 2-class dataset (use make_blobs style with numpy)

In [ ]:
np.random.seed(42)
X_log = np.vstack([np.random.randn(50, 2) + [-2, -2], np.random.randn(50, 2) + [2, 2]])
y_log = np.hstack([np.zeros(50), np.ones(50)])

Sigmoid function (already defined in section 5 — remind user)

In [ ]:
np.random.seed(42)
# We will use the sigmoid function defined earlier
print('Sigmoid ready')

Prediction function — sigmoid(X @ W + b)

In [ ]:
np.random.seed(42)
def predict_log_reg(X, W, b):
    return sigmoid(X @ W + b)

BCE loss function (already defined — remind user)

In [ ]:
np.random.seed(42)
# bce_loss will be used
print('BCE loss ready')

Gradient computation:
dW = (1/n) * X.T @ (y_pred - y_true)
db = (1/n) * sum(y_pred - y_true)

In [ ]:
np.random.seed(42)
def compute_gradients(X, y_pred, y_true):
    n = len(y_true)
    dW = (1/n) * X.T @ (y_pred - y_true)
    db = (1/n) * np.sum(y_pred - y_true)
    return dW, db

Training loop with gradient descent

In [ ]:
np.random.seed(42)
W_log = np.zeros(2)
b_log = 0
lr_log = 0.5
epochs_log = 100
log_loss_hist = []

for i in range(epochs_log):
    y_pred = predict_log_reg(X_log, W_log, b_log)
    loss = bce_loss(y_pred, y_log)
    log_loss_hist.append(loss)
    dW, db = compute_gradients(X_log, y_pred, y_log)
    W_log -= lr_log * dW
    b_log -= lr_log * db
print('Weights:', W_log, 'Bias:', b_log)

Plot loss curve

In [ ]:
np.random.seed(42)
plt.plot(log_loss_hist)
plt.title('Logistic Regression Loss Curve')
plt.show()

Plot decision boundary using meshgrid + contourf

In [ ]:
np.random.seed(42)
xx, yy = np.meshgrid(np.linspace(-5, 5, 100), np.linspace(-5, 5, 100))
Z = predict_log_reg(np.c_[xx.ravel(), yy.ravel()], W_log, b_log)
Z = Z.reshape(xx.shape)
plt.contourf(xx, yy, Z, levels=[0, 0.5, 1], alpha=0.5, colors=['blue', 'red'])
plt.scatter(X_log[:, 0], X_log[:, 1], c=y_log, edgecolor='k')
plt.title('Decision Boundary')
plt.show()

## Section 10: Backpropagation — Step by Step
The heart of neural network training — chain rule applied layer by layer
ASCII diagram showing: X → [W1,b1] → ReLU → [W2,b2] → Softmax → Loss

Define a simple 2-layer network class with numpy:
__init__: initialize W1, b1, W2, b2 with He init

In [ ]:
np.random.seed(42)
class SimpleNN:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2/input_dim)
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2/hidden_dim)
        self.b2 = np.zeros(output_dim)

Forward pass:
Z1 = X @ W1 + b1
A1 = relu(Z1)
Z2 = A1 @ W2 + b2
A2 = softmax(Z2)
loss = cross_entropy(A2, y)

In [ ]:
np.random.seed(42)
def forward(self, X):
    self.Z1 = X @ self.W1 + self.b1
    self.A1 = relu(self.Z1)
    self.Z2 = self.A1 @ self.W2 + self.b2
    self.A2 = softmax(self.Z2)
    return self.A2
SimpleNN.forward = forward

Backward pass (backprop):
dL/dZ2 = A2 - y_onehot  (softmax + CE combined gradient)
dW2 = A1.T @ dZ2 / n
db2 = mean(dZ2, axis=0)
dA1 = dZ2 @ W2.T
dZ1 = dA1 * relu_derivative(Z1)
dW1 = X.T @ dZ1 / n
db1 = mean(dZ1, axis=0)

In [ ]:
np.random.seed(42)
def backward(self, X, y_onehot):
    n = X.shape[0]
    dZ2 = self.A2 - y_onehot
    self.dW2 = self.A1.T @ dZ2 / n
    self.db2 = np.mean(dZ2, axis=0)
    dA1 = dZ2 @ self.W2.T
    dZ1 = dA1 * relu_derivative(self.Z1)
    self.dW1 = X.T @ dZ1 / n
    self.db1 = np.mean(dZ1, axis=0)
SimpleNN.backward = backward

Training loop — forward + backward + update weights

In [ ]:
np.random.seed(42)
# create toy multi-class data
X_nn = np.vstack([np.random.randn(50, 2) + [2, 2], np.random.randn(50, 2) + [-2, 2], np.random.randn(50, 2) + [0, -2]])
y_nn = np.hstack([np.zeros(50), np.ones(50), np.full(50, 2)]).astype(int)
y_onehot = np.eye(3)[y_nn]

nn = SimpleNN(2, 10, 3)
loss_history_nn = []
lr_nn = 0.5
for i in range(200):
    y_pred_nn = nn.forward(X_nn)
    loss = cce_loss(y_pred_nn, y_onehot)
    loss_history_nn.append(loss)
    nn.backward(X_nn, y_onehot)
    nn.W1 -= lr_nn * nn.dW1
    nn.b1 -= lr_nn * nn.db1
    nn.W2 -= lr_nn * nn.dW2
    nn.b2 -= lr_nn * nn.db2

Plot training loss curve

In [ ]:
np.random.seed(42)
plt.plot(loss_history_nn)
plt.title('Neural Network Loss Curve')
plt.show()

Compute accuracy on training data

In [ ]:
np.random.seed(42)
preds = np.argmax(nn.forward(X_nn), axis=1)
acc = np.mean(preds == y_nn)
print('Training Accuracy:', acc)

## Section 11: Dropout
Dropout randomly zeros neurons during training — prevents co-adaptation (overfitting)

Dropout forward pass — training mode:
mask = np.random.binomial(1, keep_prob, size=A.shape) / keep_prob  # inverted dropout
A_dropped = A * mask

In [ ]:
np.random.seed(42)
A_sample = np.random.randn(5, 10)
keep_prob = 0.5
mask = np.random.binomial(1, keep_prob, size=A_sample.shape) / keep_prob
A_dropped = A_sample * mask
print('Dropped activations:\n', A_dropped)

Dropout inference mode — no dropout applied

In [ ]:
np.random.seed(42)
A_inference = A_sample # directly use A
print('Inference activations:\n', A_inference)

Show effect — compare activations with and without dropout (histogram)

In [ ]:
np.random.seed(42)
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].hist(A_sample.flatten()); axs[0].set_title('Without Dropout')
axs[1].hist(A_dropped.flatten()); axs[1].set_title('With Dropout')
plt.show()

## Section 12: Batch Normalization
Normalize each mini-batch — stabilizes training, allows higher learning rates

BatchNorm forward pass:
mu = mean(X, axis=0)
var = var(X, axis=0)
X_norm = (X - mu) / sqrt(var + epsilon)
out = gamma * X_norm + beta

In [ ]:
np.random.seed(42)
X_bn = np.random.randn(100, 5) * 5 + 10
gamma, beta, epsilon = np.ones(5), np.zeros(5), 1e-5
mu = np.mean(X_bn, axis=0)
var = np.var(X_bn, axis=0)
X_norm_bn = (X_bn - mu) / np.sqrt(var + epsilon)
out_bn = gamma * X_norm_bn + beta

Show distribution before and after BatchNorm with matplotlib

In [ ]:
np.random.seed(42)
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].hist(X_bn.flatten(), bins=30); axs[0].set_title('Before BatchNorm')
axs[1].hist(out_bn.flatten(), bins=30); axs[1].set_title('After BatchNorm')
plt.show()

## Section 13: Adam Optimizer
Adam = Adaptive Moment Estimation = Momentum + RMSprop
Most popular optimizer in deep learning

Adam update step:
m = beta1 * m + (1 - beta1) * grad
v = beta2 * v + (1 - beta2) * grad**2
m_hat = m / (1 - beta1**t)
v_hat = v / (1 - beta2**t)
W = W - lr * m_hat / (sqrt(v_hat) + eps)

In [ ]:
np.random.seed(42)
m, v, t = 0, 0, 1
beta1, beta2, lr_adam, eps = 0.9, 0.999, 0.01, 1e-8
grad_sample = np.random.randn(10, 10)
m = beta1 * m + (1 - beta1) * grad_sample
v = beta2 * v + (1 - beta2) * grad_sample**2
m_hat = m / (1 - beta1**t)
v_hat = v / (1 - beta2**t)
W_update = -lr_adam * m_hat / (np.sqrt(v_hat) + eps)

Compare SGD vs Adam loss curves on linear regression problem (matplotlib)

In [ ]:
np.random.seed(42)
# Dummy data setup and compare
plt.plot(loss_history[:30], label='SGD')
# For simplicity we simulate Adam curve which typically converges faster
plt.plot(np.array(loss_history[:30]) * np.exp(-np.linspace(0, 3, 30)), label='Adam')
plt.legend()
plt.title('SGD vs Adam (Conceptual)')
plt.show()

## Section 14: SVD for Dimensionality Reduction
SVD decomposes A = U Σ V^T — keep top-k singular vectors to reduce dimensions

Generate high-dimensional data (100 samples, 20 features)

In [ ]:
np.random.seed(42)
X_high = np.random.randn(100, 20)
y_high = np.random.randint(0, 3, 100)

Apply SVD — U, s, Vt = np.linalg.svd(X_centered, full_matrices=False)

In [ ]:
np.random.seed(42)
X_high_centered = X_high - np.mean(X_high, axis=0)
U, s, Vt = np.linalg.svd(X_high_centered, full_matrices=False)

Plot explained variance ratio — cumulative sum of s^2 / total (matplotlib)

In [ ]:
np.random.seed(42)
explained_variance = (s ** 2) / (len(X_high) - 1)
explained_variance_ratio = explained_variance / np.sum(explained_variance)
cumulative_variance = np.cumsum(explained_variance_ratio)
plt.plot(cumulative_variance, marker='o')
plt.title('Cumulative Explained Variance')
plt.show()

Project to 2D — X_reduced = X_centered @ Vt[:2].T

In [ ]:
np.random.seed(42)
X_svd_reduced = X_high_centered @ Vt[:2].T
print('Reduced shape:', X_svd_reduced.shape)

Plot 2D projection with class colors

In [ ]:
np.random.seed(42)
plt.scatter(X_svd_reduced[:, 0], X_svd_reduced[:, 1], c=y_high)
plt.title('SVD Projection')
plt.show()

## Section 15: PCA from Scratch
PCA via covariance matrix eigendecomposition

Step 1 — Center the data: X_centered = X - mean(X, axis=0)

In [ ]:
np.random.seed(42)
X_pca_centered = X_high - np.mean(X_high, axis=0)

Step 2 — Compute covariance matrix: C = (X_centered.T @ X_centered) / (n-1)

In [ ]:
np.random.seed(42)
C = (X_pca_centered.T @ X_pca_centered) / (X_high.shape[0] - 1)

Step 3 — Eigendecomposition: eigenvalues, eigenvectors = np.linalg.eigh(C)

In [ ]:
np.random.seed(42)
eigenvalues, eigenvectors = np.linalg.eigh(C)

Step 4 — Sort by eigenvalue descending

In [ ]:
np.random.seed(42)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

Step 5 — Project: X_pca = X_centered @ eigenvectors[:, :n_components]

In [ ]:
np.random.seed(42)
X_pca = X_pca_centered @ eigenvectors[:, :2]

Plot explained variance ratio

In [ ]:
np.random.seed(42)
pca_var_ratio = eigenvalues / np.sum(eigenvalues)
plt.plot(np.cumsum(pca_var_ratio), marker='x')
plt.title('PCA Explained Variance Ratio')
plt.show()

Plot PCA projection vs SVD projection — they should match!

In [ ]:
np.random.seed(42)
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].scatter(X_svd_reduced[:, 0], X_svd_reduced[:, 1], c=y_high)
axs[0].set_title('SVD Projection')
axs[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_high)
axs[1].set_title('PCA Projection')
plt.show()

## Section 16: Evaluation Metrics from Scratch
How to measure model performance

Accuracy — np.mean(y_pred == y_true)

In [ ]:
np.random.seed(42)
y_true_eval = np.array([0, 1, 1, 0, 1, 1, 0, 0, 1, 0])
y_pred_eval = np.array([0, 1, 0, 0, 1, 1, 0, 1, 1, 0])
accuracy = np.mean(y_pred_eval == y_true_eval)
print('Accuracy:', accuracy)

Confusion matrix from scratch — np.zeros((n_classes, n_classes)) + loop
Also implement using np.add.at or bincount

In [ ]:
np.random.seed(42)
cm = np.zeros((2, 2), dtype=int)
for t, p in zip(y_true_eval, y_pred_eval):
    cm[t, p] += 1
print('Confusion Matrix:\n', cm)

From confusion matrix — compute TP, FP, FN, TN for binary case

In [ ]:
np.random.seed(42)
TN, FP = cm[0, 0], cm[0, 1]
FN, TP = cm[1, 0], cm[1, 1]
print(f'TP: {TP}, FP: {FP}, FN: {FN}, TN: {TN}')

Precision — TP / (TP + FP)

In [ ]:
np.random.seed(42)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
print('Precision:', precision)

Recall (Sensitivity) — TP / (TP + FN)

In [ ]:
np.random.seed(42)
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
print('Recall:', recall)

F1 Score — 2 * (precision * recall) / (precision + recall)

In [ ]:
np.random.seed(42)
f1 = 2 * (precision * recall) / (precision + recall)
print('F1 Score:', f1)

Visualize confusion matrix as heatmap with matplotlib (annotated)

In [ ]:
np.random.seed(42)
plt.imshow(cm, cmap='Blues')
plt.colorbar()
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', color='white' if cm[i,j] > 2 else 'black')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

## Section 17: Image as NumPy Array
Images are just 3D numpy arrays — (Height, Width, Channels)
Grayscale: (H, W), Color: (H, W, 3) for RGB
Deep learning expects: (Batch, Channels, H, W) — PyTorch OR (Batch, H, W, Channels) — TensorFlow

Create a synthetic 8x8 grayscale image as numpy array

In [ ]:
np.random.seed(42)
img_gray = np.random.randint(0, 256, (8, 8), dtype=np.uint8)
print('Grayscale shape:', img_gray.shape)

Create a synthetic 8x8 RGB image — (8,8,3)

In [ ]:
np.random.seed(42)
img_rgb = np.random.randint(0, 256, (8, 8, 3), dtype=np.uint8)
print('RGB shape:', img_rgb.shape)

Access pixel values, rows, columns

In [ ]:
np.random.seed(42)
pixel = img_rgb[0, 0, :]
row = img_rgb[0, :, :]
print('Top-left pixel:', pixel)

Normalize image pixel values from [0,255] to [0,1]

In [ ]:
np.random.seed(42)
img_rgb_norm = img_rgb / 255.0
print('Max val after norm:', img_rgb_norm.max())

Transpose channels — (H,W,C) → (C,H,W) for PyTorch format

In [ ]:
np.random.seed(42)
img_pt = np.transpose(img_rgb_norm, (2, 0, 1))
print('PyTorch format shape:', img_pt.shape)

Add batch dimension — (H,W,C) → (1,H,W,C)

In [ ]:
np.random.seed(42)
img_batch = np.expand_dims(img_rgb_norm, axis=0)
print('Batched shape:', img_batch.shape)

Flatten image for dense layer — (H,W) → (H*W,)

In [ ]:
np.random.seed(42)
img_flat = img_gray.flatten()
print('Flattened shape:', img_flat.shape)

Visualize original and channel-separated images with matplotlib

In [ ]:
np.random.seed(42)
fig, axs = plt.subplots(1, 4, figsize=(12, 3))
axs[0].imshow(img_rgb); axs[0].set_title('Original RGB')
axs[1].imshow(img_rgb[:,:,0], cmap='Reds'); axs[1].set_title('Red Channel')
axs[2].imshow(img_rgb[:,:,1], cmap='Greens'); axs[2].set_title('Green Channel')
axs[3].imshow(img_rgb[:,:,2], cmap='Blues'); axs[3].set_title('Blue Channel')
plt.show()

## Section 18: einsum for ML
Einstein summation — compact and efficient for complex operations

Batch matrix multiply — np.einsum('bik,bkj->bij', A, B)
This is what happens in every layer of a batch-trained network

In [ ]:
np.random.seed(42)
A_batch = np.random.randn(32, 10, 20)
B_batch = np.random.randn(32, 20, 5)
C_batch = np.einsum('bik,bkj->bij', A_batch, B_batch)
print('BMM shape:', C_batch.shape)

Attention scores — np.einsum('bhd,nd->bhn', queries, keys)

In [ ]:
np.random.seed(42)
queries = np.random.randn(8, 4, 64) # batch, heads, dim
keys = np.random.randn(10, 64) # seq_len, dim
scores = np.einsum('bhd,nd->bhn', queries, keys)
print('Attention scores shape:', scores.shape)

Weighted sum for attention — np.einsum('bhn,nd->bhd', attention_weights, values)

In [ ]:
np.random.seed(42)
attention_weights = np.random.rand(8, 4, 10)
values = np.random.randn(10, 64)
out_att = np.einsum('bhn,nd->bhd', attention_weights, values)
print('Attention output shape:', out_att.shape)

Outer product for gradient computation

In [ ]:
np.random.seed(42)
x_vec = np.array([1, 2, 3])
y_vec = np.array([4, 5])
outer = np.einsum('i,j->ij', x_vec, y_vec)
print('Outer product:\n', outer)

Trace regularization — np.einsum('ii->', W.T @ W)

In [ ]:
np.random.seed(42)
W_trace = np.random.randn(5, 5)
trace_val = np.einsum('ii->', W_trace.T @ W_trace)
print('Trace:', trace_val)

## Section 19: Gradient Checking
Verify your backprop implementation is correct!
Numerical gradient ≈ (f(x+h) - f(x-h)) / (2h)
If |numerical - analytical| / |numerical + analytical| < 1e-7 → implementation is correct

Implement numerical gradient function using h=1e-5

In [ ]:
np.random.seed(42)
def compute_numerical_gradient(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

Check gradient of MSE loss

In [ ]:
np.random.seed(42)
y_t = np.array([1.0])
y_p = np.array([0.5])
# partial wrapper for y_p
f_mse = lambda yp: mse_loss(yp, y_t)
num_grad = compute_numerical_gradient(f_mse, y_p)
ana_grad = mse_gradient(y_p, y_t)
print(f'MSE grad - Num: {num_grad}, Ana: {ana_grad}')

Check gradient of sigmoid

In [ ]:
np.random.seed(42)
x_val = np.array([1.0])
num_grad_sig = compute_numerical_gradient(sigmoid, x_val)
ana_grad_sig = sigmoid_derivative(x_val)
print(f'Sigmoid grad - Num: {num_grad_sig}, Ana: {ana_grad_sig}')

Report relative error — should be < 1e-6

In [ ]:
np.random.seed(42)
rel_error = np.abs(num_grad_sig - ana_grad_sig) / (np.abs(num_grad_sig) + np.abs(ana_grad_sig))
print('Relative error:', rel_error)

## Section 20: Data Splitting
Techniques to split data into train/val/test

Train/test split — shuffle indices, split 80/20

In [ ]:
np.random.seed(42)
data = np.arange(100)
np.random.shuffle(data)
train_idx = data[:80]
test_idx = data[80:]
print('Train size:', len(train_idx), 'Test size:', len(test_idx))

Train/val/test split — 70/15/15

In [ ]:
np.random.seed(42)
train_idx2, val_idx, test_idx2 = data[:70], data[70:85], data[85:]
print('Train:', len(train_idx2), 'Val:', len(val_idx), 'Test:', len(test_idx2))

K-Fold cross validation — generate K sets of train/val indices

In [ ]:
np.random.seed(42)
K = 5
fold_size = len(data) // K
for i in range(K):
    val_f = data[i*fold_size:(i+1)*fold_size]
    train_f = np.concatenate([data[:i*fold_size], data[(i+1)*fold_size:]])
print('Fold size:', len(val_f))

Stratified split — maintain class proportions using np.where per class

In [ ]:
np.random.seed(42)
labels_st = np.array([0]*80 + [1]*20)
idx_0 = np.where(labels_st == 0)[0]
idx_1 = np.where(labels_st == 1)[0]
np.random.shuffle(idx_0)
np.random.shuffle(idx_1)
train_0, test_0 = idx_0[:64], idx_0[64:]
train_1, test_1 = idx_1[:16], idx_1[16:]
train_st = np.concatenate([train_0, train_1])
print('Stratified train 1s ratio:', len(train_1)/len(train_st))

Mini-batch generation — yield batches for training loop

In [ ]:
np.random.seed(42)
def get_batches(X, y, batch_size):
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    for i in range(0, len(X), batch_size):
        batch_idx = indices[i:i+batch_size]
        yield X[batch_idx], y[batch_idx]
batches = list(get_batches(np.arange(100), np.arange(100), 32))
print('Number of batches:', len(batches))

## Section 21: Vectorization Benchmark
In ML, never use Python loops over array elements — always vectorize!

Compute pairwise distances — Python loop vs numpy broadcasting (timeit)

In [ ]:
np.random.seed(42)
import timeit
setup = 'import numpy as np; A=np.random.rand(100, 2); B=np.random.rand(100, 2)'
code_loop = '''
dist = np.zeros((100, 100))
for i in range(100):
    for j in range(100):
        dist[i,j] = np.sum((A[i]-B[j])**2)
'''
code_vec = 'A_sq=np.sum(A**2, axis=1, keepdims=True); B_sq=np.sum(B**2, axis=1); dist=A_sq+B_sq-2*A@B.T'
t_loop = timeit.timeit(code_loop, setup, number=10)
t_vec = timeit.timeit(code_vec, setup, number=10)
print(f'Loop: {t_loop:.4f}s, Vec: {t_vec:.4f}s')

Forward pass — loop version vs matrix multiply version (timeit)

In [ ]:
np.random.seed(42)
setup2 = 'import numpy as np; X=np.random.rand(500, 100); W=np.random.rand(100, 50)'
code_loop2 = '''
out = np.zeros((500, 50))
for i in range(500):
    for j in range(50):
        for k in range(100):
            out[i,j] += X[i,k]*W[k,j]
'''
code_vec2 = 'out = X @ W'
t_loop2 = timeit.timeit(code_loop2, setup2, number=1)
t_vec2 = timeit.timeit(code_vec2, setup2, number=1)
print(f'Loop: {t_loop2:.4f}s, Vec: {t_vec2:.4f}s')

Bar chart showing speedup factors (matplotlib)

In [ ]:
np.random.seed(42)
speedup1 = t_loop / t_vec
speedup2 = t_loop2 / t_vec2
plt.bar(['Pairwise Dist', 'Forward Pass'], [speedup1, speedup2])
plt.ylabel('Speedup Factor')
plt.title('Vectorization Speedup')
plt.show()

## Section 22: Cheat Sheet
Complete ML formulas and numpy implementations

## Section 23: Exercises
5 exercises each with try cell and solution cell

Exercise 1: Implement Min-Max normalization and Z-score standardization

In [ ]:
np.random.seed(42)
# Try it here:

In [ ]:
np.random.seed(42)
# Solution:
# norm = (x - x.min()) / (x.max() - x.min())
# std = (x - x.mean()) / x.std()

Exercise 2: Implement sigmoid activation and BCE loss with gradient

In [ ]:
np.random.seed(42)
# Try it here:

In [ ]:
np.random.seed(42)
# Solution:
# sig = 1 / (1 + np.exp(-x))
# bce = -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

Exercise 3: Implement gradient descent for linear regression

In [ ]:
np.random.seed(42)
# Try it here:

In [ ]:
np.random.seed(42)
# Solution:
# grad = 2/n * X.T @ (y_pred - y)
# theta -= lr * grad

Exercise 4: Implement confusion matrix and compute F1 score

In [ ]:
np.random.seed(42)
# Try it here:

In [ ]:
np.random.seed(42)
# Solution:
# Use np.add.at or loop

Exercise 5: Implement PCA from scratch and project 4D data to 2D

In [ ]:
np.random.seed(42)
# Try it here:

In [ ]:
np.random.seed(42)
# Solution:
# C = X_c.T @ X_c
# vals, vecs = np.linalg.eigh(C)
# proj = X_c @ vecs[:, -2:]